<div class="alert alert-block alert-info">
    import <b>Libraries</b>
</div>

In [12]:

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

<div class="alert alert-block alert-info">
    <b>LOAD DATASET</b>
</div>

In [13]:
train= pd.read_csv("train.csv")
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [14]:
test = pd.read_csv("test.csv")
test.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


<div class="alert alert-block alert-info">
    <b>LOG TRANSFORM TARGET</b>
</div>

In [15]:
train["SalePrice_log"] = np.log1p(train["SalePrice"])

<div class="alert alert-block alert-info">
    <b>FEATURE SELECTION</b>
</div>

In [ ]:
num_features = [
    'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
    'MasVnrArea', 'BsmtFinSF1', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
    'GrLivArea', 'FullBath', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars',
    'GarageArea'
]

cat_features = [
    'Neighborhood', 'ExterQual', 'KitchenQual', 'Foundation',
    'BsmtQual', 'GarageFinish', 'SaleCondition', 'HouseStyle'
]

num_cols = [c for c in num_features if c in train.columns]
cat_cols = [c for c in cat_features if c in train.columns]

X = train[num_cols + cat_cols]
y = train["SalePrice_log"]

X_test = test[num_cols + cat_cols]

<div class="alert alert-block alert-info">
    <b>PREPROCESSING</b>
</div>

In [17]:
num_transformer = SimpleImputer(strategy='median')

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

<div class="alert alert-block alert-info">
    <b>MODEL</b>
</div>

In [18]:
model = xgb.XGBRegressor(
    n_estimators=1200,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=42,
    reg_alpha=0.1,
    n_jobs=-1
)

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])


<div class="alert alert-block alert-info">
    <b>VALIDATION</b>
</div>

In [19]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

full_pipeline.fit(X_train, y_train)

val_preds_log = full_pipeline.predict(X_val)

rmsle = np.sqrt(mean_absolute_error(y_val, val_preds_log))
print(f"Validation RMSLE: {rmsle:.4f}")

Validation RMSLE: 0.2970


<div class="alert alert-block alert-info">
    <b>TRAIN</b>
</div>

In [20]:
full_pipeline.fit(X, y)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


<div class="alert alert-block alert-info">
    <b>PREDICT ON TEST SET</b>
</div>

In [21]:
test_preds_log = full_pipeline.predict(X_test)
test_preds = np.expm1(test_preds_log)

<div class="alert alert-block alert-info">
    <b>FINAL SUBMISSION</b>
</div>

In [22]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": test_preds
})
submission.to_csv("submission.csv", index=False)

print("File created successfully.")

File created successfully.
